# 04 — Evaluate Supervisor Agent

This notebook evaluates the **Supply Chain Supervisor Agent** using MLflow Agent Evaluation.

We benchmark:
* **Latency** — end-to-end response time (mean, median, P95) across question categories
* **Tool-Call Correctness** — did the supervisor route to the correct Genie space?
* **Tool-Call Efficiency** — no redundant calls to irrelevant sub-agents?
* **Response Quality** — accurate, specific, actionable answers?

**Prerequisites:** Run notebooks 01–03 first to create data, Genie spaces, and the supervisor agent.

In [0]:
%pip install databricks-sdk openai "mlflow>=3.12.0" pandas --upgrade --quiet
dbutils.library.restartPython()

In [0]:
"""Setup: imports, configuration, MLflow experiment, and OpenAI client for the supervisor endpoint."""
import os
import time
import json
import pandas as pd
import mlflow
from openai import OpenAI
from databricks.sdk import WorkspaceClient

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CONFIGURATION — Only change these if you need a different setup           │
# └──────────────────────────────────────────────────────────────────────────┘

# LLM model for the MLflow judges. Any Foundation Model API endpoint works.
# Format: "databricks:/<endpoint-name>"
JUDGE_MODEL = "databricks:/databricks-claude-sonnet-4"

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ AUTO-DISCOVERED — No changes needed below this line                      │
# └──────────────────────────────────────────────────────────────────────────┘

w = WorkspaceClient()

# Current user (for experiment path and endpoint filtering)
_username = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
)

# Discover the supervisor endpoint: filter mas-* endpoints created by current user.
# AgentBricks naming convention: mas-<first-8-chars-of-agent-id>-endpoint
_my_endpoints = [
    ep.name
    for ep in w.serving_endpoints.list()
    if ep.name.startswith("mas-") and ep.name.endswith("-endpoint") and ep.creator == _username
]

if len(_my_endpoints) == 1:
    ENDPOINT_NAME = _my_endpoints[0]
elif len(_my_endpoints) > 1:
    # Multiple supervisor agents owned by this user — pick the most recently created
    # or override manually: ENDPOINT_NAME = "mas-XXXXXXXX-endpoint"
    print(f"Found {len(_my_endpoints)} supervisor endpoints: {_my_endpoints}")
    print("Using the first one. Override ENDPOINT_NAME above if needed.")
    ENDPOINT_NAME = _my_endpoints[0]
else:
    raise ValueError(
        f"No mas-*-endpoint found for user '{_username}'. "
        f"Run notebook 03 first to create the supervisor agent."
    )

# Derive the short agent ID from the endpoint name
AGENT_ID = ENDPOINT_NAME.replace("mas-", "").replace("-endpoint", "")

# Host and token (from current notebook session)
HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
TOKEN = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

# MLflow experiment — scoped to the current user's home directory
EXPERIMENT_NAME = f"/Users/{_username}/supply-chain-supervisor-eval"
mlflow.set_experiment(EXPERIMENT_NAME)

# OpenAI-compatible client for the supervisor endpoint
client = OpenAI(api_key=TOKEN, base_url=f"{HOST}/serving-endpoints")

print(f"Endpoint:    {ENDPOINT_NAME}")
print(f"Agent ID:    {AGENT_ID}")
print(f"Experiment:  {EXPERIMENT_NAME}")
print(f"Judge Model: {JUDGE_MODEL}")

## Evaluation Dataset

We define 12 evaluation questions spanning three categories:
* **Procurement** (5) — routed to `procurement_inventory` Genie space
* **Logistics** (5) — routed to `logistics_fulfillment` Genie space
* **Cross-domain** (2) — requires both tools with synthesis

Each question has an expected tool routing and description of what a correct answer looks like.

In [0]:
"""Define the evaluation dataset with questions, expected routing, and descriptions."""

eval_cases = [
    # Procurement domain
    {
        "question": "Which supplier has the highest total spend?",
        "expected_tool": "procurement_inventory",
        "category": "procurement",
        "description": "Should query procurement_metrics for total spend by supplier",
    },
    {
        "question": "Are any materials below their reorder point at warehouse WH-EAST?",
        "expected_tool": "procurement_inventory",
        "category": "procurement",
        "description": "Should check inventory table for items below reorder_point",
    },
    {
        "question": "What is the average lead time for suppliers in China?",
        "expected_tool": "procurement_inventory",
        "category": "procurement",
        "description": "Should query procurement_metrics filtering by country=China",
    },
    {
        "question": "How many purchase orders are currently in transit?",
        "expected_tool": "procurement_inventory",
        "category": "procurement",
        "description": "Should count POs with status In Transit",
    },
    {
        "question": "Which material category has the highest spend?",
        "expected_tool": "procurement_inventory",
        "category": "procurement",
        "description": "Should aggregate spend by material_category",
    },
    # Logistics domain
    {
        "question": "What is the on-time delivery rate for air freight?",
        "expected_tool": "logistics_fulfillment",
        "category": "logistics",
        "description": "Should query logistics_metrics filtering transport_mode=Air",
    },
    {
        "question": "Which carrier has the most delayed shipments?",
        "expected_tool": "logistics_fulfillment",
        "category": "logistics",
        "description": "Should query shipments/logistics_metrics for delays by carrier",
    },
    {
        "question": "What is the average shipping cost per kilogram by carrier?",
        "expected_tool": "logistics_fulfillment",
        "category": "logistics",
        "description": "Should calculate avg cost_per_kg from logistics_metrics",
    },
    {
        "question": "Show me all shipments currently in transit",
        "expected_tool": "logistics_fulfillment",
        "category": "logistics",
        "description": "Should filter shipments by status In Transit",
    },
    {
        "question": "What is the longest route by distance?",
        "expected_tool": "logistics_fulfillment",
        "category": "logistics",
        "description": "Should query routes table ordered by distance_km",
    },
    # Cross-domain
    {
        "question": "Give me a supply chain health summary across procurement and logistics",
        "expected_tool": "both",
        "category": "cross-domain",
        "description": "Should call both tools and synthesize",
    },
    {
        "question": "Compare on-time rates between suppliers and carriers",
        "expected_tool": "both",
        "category": "cross-domain",
        "description": "Should get on-time from procurement_metrics and logistics_metrics",
    },
]

eval_df = pd.DataFrame(eval_cases)
print(f"Evaluation dataset: {len(eval_df)} questions")
print(f"  Procurement: {len(eval_df[eval_df['category'] == 'procurement'])}")
print(f"  Logistics:   {len(eval_df[eval_df['category'] == 'logistics'])}")
print(f"  Cross-domain: {len(eval_df[eval_df['category'] == 'cross-domain'])}")
display(eval_df)

## Run Agent & Collect Traces with Latency

We invoke the supervisor agent for each evaluation question, capturing:
* Full response text
* End-to-end latency (includes supervisor routing + Genie space SQL generation + execution)
* Response ID for trace correlation

In [0]:
"""Helper function to invoke the supervisor agent and capture response + latency + tool calls."""


def invoke_supervisor(question: str) -> dict:
    """Invoke the supervisor agent and capture response, latency, and tool call metadata.

    Args:
        question: The natural language question to send to the supervisor.

    Returns:
        Dict with response text, latency_seconds, response_id, and tools_called list.
    """
    start_time = time.time()

    response = client.responses.create(
        model=ENDPOINT_NAME,
        input=[{"role": "user", "content": question}],
    )

    latency = time.time() - start_time

    # Extract response text and tool calls from output items
    response_text = ""
    tools_called = []
    for item in response.output:
        if hasattr(item, "content"):
            for content_block in item.content:
                if hasattr(content_block, "text"):
                    response_text += content_block.text
        # Capture function_call items (tool routing metadata)
        if hasattr(item, "type") and item.type == "function_call":
            tools_called.append(item.name)

    return {
        "response": response_text,
        "latency_seconds": latency,
        "response_id": response.id,
        "tools_called": tools_called,
    }

In [0]:
"""Run all evaluation questions through the supervisor agent and collect results."""

print("Running evaluation questions through supervisor agent...")
print("=" * 60)

results = []
for idx, row in eval_df.iterrows():
    print(f"\n[{idx + 1}/{len(eval_df)}] {row['question'][:60]}...")
    try:
        result = invoke_supervisor(row["question"])
        result["question"] = row["question"]
        result["expected_tool"] = row["expected_tool"]
        result["category"] = row["category"]
        result["description"] = row["description"]
        result["status"] = "success"
        print(f"  ✓ Latency: {result['latency_seconds']:.1f}s | Response: {result['response'][:80]}...")
    except Exception as e:
        result = {
            "question": row["question"],
            "expected_tool": row["expected_tool"],
            "category": row["category"],
            "description": row["description"],
            "response": f"ERROR: {str(e)}",
            "latency_seconds": None,
            "response_id": None,
            "status": "error",
        }
        print(f"  ✗ Error: {str(e)[:80]}")
    results.append(result)

results_df = pd.DataFrame(results)
print(f"\n{'=' * 60}")
print(
    f"Completed: {len(results_df[results_df['status'] == 'success'])} success, "
    f"{len(results_df[results_df['status'] == 'error'])} errors"
)

## Latency Benchmarks

Analyze end-to-end latency across question categories. For a managed supervisor → Genie space chain,
expect higher latency than direct SQL (supervisor routing + Genie SQL generation + warehouse execution).

In [0]:
"""Latency benchmark analysis: overall stats and breakdowns by category and expected tool."""

successful = results_df[results_df["status"] == "success"]

print("=" * 60)
print("LATENCY BENCHMARK SUMMARY")
print("=" * 60)

# Overall stats
print(f"\nOverall (n={len(successful)}):")
print(f"  Mean:   {successful['latency_seconds'].mean():.2f}s")
print(f"  Median: {successful['latency_seconds'].median():.2f}s")
print(f"  P95:    {successful['latency_seconds'].quantile(0.95):.2f}s")
print(f"  Min:    {successful['latency_seconds'].min():.2f}s")
print(f"  Max:    {successful['latency_seconds'].max():.2f}s")

# By category
print(f"\nBy Category:")
for cat, group in successful.groupby("category"):
    print(
        f"  {cat}: mean={group['latency_seconds'].mean():.2f}s, "
        f"median={group['latency_seconds'].median():.2f}s (n={len(group)})"
    )

# By expected tool
print(f"\nBy Expected Tool:")
for tool, group in successful.groupby("expected_tool"):
    print(
        f"  {tool}: mean={group['latency_seconds'].mean():.2f}s, "
        f"median={group['latency_seconds'].median():.2f}s (n={len(group)})"
    )

In [0]:
"""Visualize a single MLflow trace to understand the supervisor execution flow.

This cell shows EXACTLY what happens when you ask the supervisor a question:
1. Your question hits the serving endpoint (CHAT_MODEL span)
2. The supervisor decides which Genie space tool to call (function_call in output)
3. The Genie space generates SQL, executes it, and returns data
4. The supervisor synthesizes the final answer

Latency is captured as wall-clock time around the entire request (time.time() before/after).
The MLflow trace lets us inspect the internal structure of that request.
"""

import mlflow

# Enable autolog so the OpenAI client call is captured as an MLflow trace
mlflow.openai.autolog()

# Make one traced call to the supervisor
print("\U0001f50d Calling supervisor with MLflow tracing enabled...")
print("   Question: 'Which supplier has the highest total spend?'")
print()

traced_response = client.responses.create(
    model=ENDPOINT_NAME,
    input=[{"role": "user", "content": "Which supplier has the highest total spend?"}],
)

# Retrieve the trace
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
client_mlflow = mlflow.tracking.MlflowClient()
traces = client_mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=1,
    order_by=["timestamp_ms DESC"],
)
trace = traces[0]

# === VISUALIZE THE TRACE STRUCTURE ===
hr = "\u2500" * 62
box_top = "\u250c" + "\u2500" * 62 + "\u2510"
box_bot = "\u2514" + "\u2500" * 62 + "\u2518"

print("=" * 70)
print("\U0001f4cd MLflow TRACE VISUALIZATION")
print("=" * 70)
print(f"\n  Trace ID: {trace.info.trace_id}")
print(f"  Duration: {trace.info.execution_duration / 1000:.2f}s")
print()

# Show the span (what MLflow captured)
span = trace.data.spans[0]
print(f"  {box_top}")
print(f"  \u2502  Span: {span.name} (type: {span.span_type})" + " " * 20 + "\u2502")
print(f"  \u2502  This is the single API call to the supervisor endpoint.    \u2502")
print(f"  {box_bot}")

# Parse the output items to show the execution flow
output_items = span.outputs.get("output", [])
print(f"\n  \U0001f4e8 EXECUTION FLOW (parsed from response output items):")
print(f"  {hr}")

step = 1
for item in output_items:
    item_type = item.get("type", "unknown")

    if item_type == "function_call":
        tool_name = item.get("name", "?")
        args = item.get("arguments", "{}")
        print(f"\n  Step {step}: \U0001f6e0\ufe0f  TOOL CALL")
        print(f"         Tool: {tool_name}")
        print(f"         Args: {args[:80]}")
        print(f"         (Supervisor routed your question to this Genie space)")
        step += 1

    elif item_type == "message":
        content = item.get("content", [])
        text = content[0].get("text", "") if content else ""
        if text.startswith("|") or "supplier_name" in text.lower():
            print(f"\n  Step {step}: \U0001f4ca DATA RETURNED")
            lines = text.strip().split("\n")[:4]
            for line in lines:
                print(f"         {line[:70]}")
            total_lines = len(text.strip().split("\n"))
            if total_lines > 4:
                print(f"         ... ({total_lines} rows total)")
            print(f"         (Genie space generated SQL, ran it, returned results)")
            step += 1
        elif not text.startswith("<name>") and len(text) > 20:
            print(f"\n  Step {step}: \u2705 FINAL ANSWER")
            print(f"         {text[:100]}")
            print(f"         (Supervisor synthesized the data into a human-readable response)")
            step += 1

print(f"\n  {hr}")
duration_s = trace.info.execution_duration / 1000
print(f"\n  \u23f1\ufe0f  HOW LATENCY IS MEASURED:")
print(f"     start = time.time()")
print(f"     response = client.responses.create(...)  # <-- ALL steps above happen here")
print(f"     latency = time.time() - start")
print()
print(f"     The {duration_s:.1f}s total includes:")
print(f"       \u2022 Supervisor LLM routing decision (~1-2s)")
print(f"       \u2022 Genie space SQL generation (~5-10s)")
print(f"       \u2022 SQL warehouse query execution (~5-15s)")
print(f"       \u2022 Supervisor answer synthesis (~2-3s)")
print(f"       \u2022 Network round-trips between services")

# Disable autolog to avoid tracing the eval calls
mlflow.openai.autolog(disable=True)

## MLflow Evaluation

We use a **hybrid scoring approach** optimized for managed Supervisor Agents:

### Why NOT built-in `ToolCallCorrectness`?
The built-in scorer requires `TOOL`-type spans in the MLflow trace. Our managed supervisor
executes tools **server-side** — autolog only captures a single `CHAT_MODEL` span. The scorer
says "no tools were called" even though `function_call` items ARE in the response output.

### Our approach:
1. **Deterministic code scorer** — Parses `function_call` items from the raw response to extract
   the actual tool name called, then compares against `expected_tool`. No LLM needed, 100% reliable.
2. **Custom LLM judges** — Assess subjective quality dimensions (efficiency, response quality)
   that require reasoning over the response content.

In [0]:
"""Define scorers: deterministic routing scorer + custom LLM judges."""

from typing import Literal
from mlflow.genai.judges import make_judge

JUDGE_MODEL = "databricks:/databricks-claude-sonnet-4"

# ═══════════════════════════════════════════════════════════════════════════════
# SCORER 1: Deterministic Tool-Call Correctness (code-based, no LLM)
# ═══════════════════════════════════════════════════════════════════════════════
# This directly validates which tool was called vs expected — extracted from the
# function_call items in the Responses API output. 100% reliable, zero latency.


def score_routing_correctness(row: dict) -> dict:
    """Deterministic scorer: did the supervisor route to the correct tool?

    Compares the actual function_call names from the response against expected_tool.
    Returns pass/fail with explanation.
    """
    tools_called = row.get("tools_called", [])
    expected = row.get("expected_tool", "")

    if expected == "both":
        # Cross-domain: should call both tools
        called_set = set(tools_called)
        expected_set = {"procurement_inventory", "logistics_fulfillment"}
        if expected_set.issubset(called_set):
            return {"pass": True, "reason": f"Correctly called both tools: {tools_called}"}
        else:
            missing = expected_set - called_set
            return {"pass": False, "reason": f"Expected both tools, missing: {missing}. Called: {tools_called}"}
    else:
        # Single-domain: should call exactly the expected tool
        if expected in tools_called:
            return {"pass": True, "reason": f"Correctly routed to {expected}"}
        else:
            return {"pass": False, "reason": f"Expected {expected}, but called: {tools_called}"}


print("✓ Deterministic routing scorer defined (score_routing_correctness)")
print("  Parses function_call items from raw response — no LLM needed")

# ═══════════════════════════════════════════════════════════════════════════════
# SCORER 2: LLM Judge — Tool-Call Efficiency
# ═══════════════════════════════════════════════════════════════════════════════
tool_call_efficiency_judge = make_judge(
    name="tool_call_efficiency",
    instructions="""You are evaluating whether an AI agent's response indicates efficient tool usage (no redundant or unnecessary operations).

Given:
- User question: {{ inputs }}
- Agent response: {{ outputs }}

Evaluate efficiency:
- A single-domain question (about procurement OR logistics, not both) should produce a focused response from one domain only -> efficient
- A cross-domain question should produce a synthesized response covering both domains -> efficient
- If a single-domain question produces responses mixing irrelevant domain data -> inefficient
- If the response is excessively verbose with repeated/redundant information -> inefficient

Return "yes" if the response indicates efficient tool usage, "no" if it shows redundancy or unnecessary work.""",
    feedback_value_type=Literal["yes", "no"],
    model=JUDGE_MODEL,
)

# ═══════════════════════════════════════════════════════════════════════════════
# SCORER 3: LLM Judge — Response Quality
# ═══════════════════════════════════════════════════════════════════════════════
response_quality_judge = make_judge(
    name="response_quality",
    instructions="""You are evaluating the quality of a Supply Chain AI agent's response.

Given:
- User question: {{ inputs }}
- Agent response: {{ outputs }}

Evaluate on these criteria:
1. RELEVANCE: Does the response directly address the question asked?
2. SPECIFICITY: Does it provide specific data points, numbers, or names (not vague generalities)?
3. ACTIONABILITY: Is the information presented clearly and usably?
4. COMPLETENESS: Does it fully answer what was asked?

Return "yes" if the response meets at least 3 of 4 criteria well, "no" otherwise.""",
    feedback_value_type=Literal["yes", "no"],
    model=JUDGE_MODEL,
)

print("✓ LLM judges defined:")
print(f"  Model: {JUDGE_MODEL}")
print("  1. tool_call_efficiency - No redundant tool calls?")
print("  2. response_quality - Accurate, specific, actionable?")

In [0]:
"""Run evaluation: deterministic routing scorer + MLflow LLM judges."""

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Part 1: Deterministic Routing Correctness (instant, no LLM cost)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("DETERMINISTIC ROUTING EVALUATION")
print("=" * 60)

routing_results = []
for _, row in results_df[results_df["status"] == "success"].iterrows():
    score = score_routing_correctness(row)
    routing_results.append({
        "question": row["question"],
        "expected_tool": row["expected_tool"],
        "tools_called": row["tools_called"],
        "routing_pass": score["pass"],
        "routing_reason": score["reason"],
    })

routing_df = pd.DataFrame(routing_results)
pass_count = routing_df["routing_pass"].sum()
total = len(routing_df)
print(f"\n  Routing Correctness: {pass_count}/{total} ({100 * pass_count / total:.0f}%)")

# Show failures
failures = routing_df[~routing_df["routing_pass"]]
if len(failures) > 0:
    print(f"\n  Misrouted questions ({len(failures)}):")
    for _, f in failures.iterrows():
        print(f"    ✗ {f['question'][:55]}")
        print(f"      {f['routing_reason']}")
else:
    print("  ✓ All questions routed to the correct tool!")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Part 2: LLM Judge Evaluation (efficiency + quality)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(f"\n\nLLM JUDGE EVALUATION")
print("=" * 60)

# Prepare evaluation data for MLflow
eval_data = []
for _, row in results_df[results_df["status"] == "success"].iterrows():
    eval_data.append(
        {
            "inputs": {"question": row["question"]},
            "outputs": {"response": row["response"]},
        }
    )

eval_dataset = pd.DataFrame(eval_data)
print(f"  Dataset: {len(eval_dataset)} rows")

# Run MLflow evaluation
with mlflow.start_run(run_name="supervisor_eval_v2_hybrid") as run:
    # Log parameters
    mlflow.log_param("endpoint_name", ENDPOINT_NAME)
    mlflow.log_param("agent_id", AGENT_ID)
    mlflow.log_param("num_questions", len(eval_data))
    mlflow.log_param("mean_latency_s", round(successful["latency_seconds"].mean(), 2))
    mlflow.log_param("p95_latency_s", round(successful["latency_seconds"].quantile(0.95), 2))
    mlflow.log_param("routing_correctness_pct", round(100 * pass_count / total, 1))
    mlflow.log_metric("routing_correctness", pass_count / total)

    eval_result = mlflow.genai.evaluate(
        data=eval_dataset,
        scorers=[
            tool_call_efficiency_judge,
            response_quality_judge,
        ],
    )

    print(f"\n  MLflow Run ID: {run.info.run_id}")
    print(f"  Evaluation Metrics:")
    for metric_name, metric_value in eval_result.metrics.items():
        print(f"    {metric_name}: {metric_value}")

print("\n✓ Evaluation complete.")

## Evaluation Results Summary

Combined view of latency benchmarks and judge scores with actionable recommendations.

In [0]:
"""Display comprehensive evaluation summary: latency, routing, judge scores, and recommendations."""

import mlflow

print("=" * 60)
print("SUPERVISOR AGENT EVALUATION SUMMARY")
print("=" * 60)

# Latency summary
print("\n\U0001f4ca LATENCY")
print(f"  Mean:   {successful['latency_seconds'].mean():.2f}s")
print(f"  Median: {successful['latency_seconds'].median():.2f}s")
print(f"  P95:    {successful['latency_seconds'].quantile(0.95):.2f}s")

# Deterministic routing score
print(f"\n\U0001f3af ROUTING CORRECTNESS (deterministic)")
pass_count = routing_df["routing_pass"].sum()
total = len(routing_df)
pct = 100 * pass_count / total
emoji = "\u2713" if pct == 100 else "\u26a1" if pct >= 80 else "\u26a0\ufe0f"
print(f"  {emoji} {pass_count}/{total} ({pct:.0f}%) \u2014 verified via function_call metadata")

# Retrieve LLM judge scores from MLflow traces (latest 12 with assessments)
print(f"\n\U0001f9d1\u200d\u2696\ufe0f LLM JUDGE SCORES")
client_mlflow = mlflow.tracking.MlflowClient()
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
traces = client_mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=12,
    order_by=["timestamp_ms DESC"],
)

assessment_results = []
for t in traces:
    for a in t.info.assessments:
        if a.name in ("tool_call_efficiency", "response_quality"):
            assessment_results.append(
                {"scorer": a.name, "value": a.feedback.value if a.feedback else None}
            )

if assessment_results:
    assess_df = pd.DataFrame(assessment_results)
    for scorer, group in assess_df.groupby("scorer"):
        yes_count = len(group[group["value"] == "yes"])
        total_g = len(group)
        pct_g = 100 * yes_count / total_g if total_g > 0 else 0
        emoji_g = "\u2713" if pct_g >= 90 else "\u26a1" if pct_g >= 70 else "\u26a0\ufe0f"
        print(f"  {emoji_g} {scorer}: {yes_count}/{total_g} ({pct_g:.0f}%)")
else:
    print("  (assessments still processing \u2014 re-run this cell in a moment)")

# Per-question detail table with routing result
print(f"\n\U0001f4cb PER-QUESTION RESULTS")
detail_df = results_df[
    ["question", "category", "expected_tool", "tools_called", "latency_seconds", "status"]
].copy()
detail_df["question"] = detail_df["question"].str[:45] + "..."
detail_df = detail_df.merge(
    routing_df[["question", "routing_pass"]].assign(
        question=routing_df["question"].str[:45] + "..."
    ),
    on="question",
    how="left",
)
display(detail_df)

# Recommendations
print(f"\n\U0001f4a1 RECOMMENDATIONS")
mean_latency = successful["latency_seconds"].mean()
if mean_latency > 30:
    print("  \u26a0\ufe0f  High latency. Genie spaces add SQL generation + warehouse overhead.")
    print("     - Expected for managed supervisor \u2192 Genie space chains")
    print("     - Cross-domain questions (~84s) call both tools sequentially")
elif mean_latency > 15:
    print("  \u26a1 Moderate latency. Expected for multi-hop orchestration.")
else:
    print("  \u2713 Latency within acceptable range.")

routing_failures = routing_df[~routing_df["routing_pass"]]
if len(routing_failures) > 0:
    print(f"\n  \u26a0\ufe0f  Routing issues ({len(routing_failures)}):")
    for _, f in routing_failures.iterrows():
        print(f"     - \"{f['question'][:50]}\"")
        print(f"       Fix: Add guideline in Examples tab for this pattern")
else:
    print("\n  \u2713 Perfect routing \u2014 all questions sent to the correct Genie space.")

## Next Steps

* **Improve routing**: Go to supervisor configuration → Examples tab → add guidelines based on misrouted questions
* **Add human review**: Share MLflow experiment with SMEs to rate responses in the Review App
* **Monitor production**: Set up Lakehouse Monitoring on the inference table for ongoing drift detection
* **Iterate on judges**: Refine judge instructions based on false positives/negatives observed above
* **Expand dataset**: Add edge cases (ambiguous questions, out-of-scope queries) to stress-test routing